<a href="https://colab.research.google.com/github/youma-code/qqq/blob/main/%E3%83%9A%E3%83%BC%E3%83%91%E3%83%BC%E3%83%88%E3%83%AC%E3%83%BC%E3%83%89.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# Install the 'ta' library if it's not already installed
!pip install ta-lib

print("TA library installation attempted.")


TA library installation attempted.


In [15]:
!pip install ta

# QQQ Paper Trading Simulation with Optuna Optimized Parameters

## 1. Parameter Loading

In [16]:
import json

# Define the path to your best_params.json file
params_file_path = '/content/drive/MyDrive/optuna_trading/best_params.json'

try:
    with open(params_file_path, 'r') as f:
        best_params = json.load(f)
    print(f"Successfully loaded optimized parameters from {params_file_path}:")
    for param, value in best_params.items():
        print(f"  {param}: {value}")
except FileNotFoundError:
    print(f"Error: The file {params_file_path} was not found. Please ensure the path is correct and the file exists. Using empty parameters.")
    best_params = {} # Ensure best_params is defined as an empty dict on error
except json.JSONDecodeError:
    print(f"Error: Could not decode JSON from {params_file_path}. Please check file integrity. Using empty parameters.")
    best_params = {}


Error: The file /content/drive/MyDrive/optuna_trading/best_params.json was not found. Please ensure the path is correct and the file exists. Using empty parameters.


## 2. Data Acquisition, Feature Generation, and Preprocessing

In [17]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta # A useful library for technical analysis indicators

# --- IMPORTANT: Replicate your backtest's feature generation and preprocessing exactly ---

def generate_features(df):
    """
    Generates technical indicators and other features required for the trading strategy.
    This function *must* exactly match the feature generation used during Optuna optimization.
    """
    # Example features (replace with your actual backtest features):
    df['SMA_10'] = ta.trend.sma_indicator(df['Close'], window=10)
    df['SMA_50'] = ta.trend.sma_indicator(df['Close'], window=50)
    df['RSI'] = ta.momentum.rsi(df['Close'], window=14)
    macd = ta.trend.macd(df['Close'])
    df['MACD'] = macd
    df['MACD_Signal'] = ta.trend.macd_signal(df['Close'])
    df['ATR'] = ta.volatility.average_true_range(df['High'], df['Low'], df['Close'], window=14)

    # Add more features as per your backtest logic (e.g., volatility, volume indicators, custom metrics)
    # df['Custom_Feature'] = ...

    # Ensure no future leakage during feature generation (e.g., using `shift` if needed)
    # For this real-time simulation, we will apply this function to historical data
    # and then to the latest bar in the signal generation step.

    return df.dropna().copy()

def calculate_regime(df):
    """
    Calculates the market regime. This function *must* exactly match the regime calculation
    used during Optuna optimization and backtesting.
    """
    # Example regime calculation (replace with your actual backtest logic):
    # For simplicity, let's say a regime is 'bullish' if SMA_10 > SMA_50, else 'bearish'
    # Or 'sideways' if ATR is low and price change is minimal, etc.
    df['Regime'] = 'Neutral'
    # This is a placeholder. You need to implement your actual regime logic here.
    # Example: Based on price relative to moving averages, ADX, volatility, etc.
    # For example, using SMA crossover:
    df.loc[df['SMA_10'] > df['SMA_50'], 'Regime'] = 'Bullish'
    df.loc[df['SMA_10'] <= df['SMA_50'], 'Regime'] = 'Bearish'

    # Further define your 'sideways' logic if applicable
    # For instance, if ADX < 25, it might be a sideways market
    # df['ADX'] = ta.trend.adx(df['High'], df['Low'], df['Close'], window=14)
    # df.loc[df['ADX'] < 25, 'Regime'] = 'Sideways'

    return df


# --- Fetch QQQ Data ---
ticker = 'QQQ'
# Fetch data up to the current date for simulation. Extend history for features needing lookback.
# Ensure enough historical data for all indicators to be calculated.
end_date = pd.Timestamp.now(tz='UTC').date() # Current date
start_date = end_date - pd.DateOffset(years=2) # e.g., 2 years of data

print(f"Fetching {ticker} data from {start_date} to {end_date}...")
try:
    # Use 'interval="1d"' for daily data, as specified
    df = yf.download(ticker, start=start_date, end=end_date, interval="1d")
    if df.empty:
        raise ValueError("No data downloaded. Check ticker symbol or date range.")
    print("Data downloaded successfully.")

    # Apply feature generation
    df_processed = generate_features(df)

    # Apply regime calculation
    df_processed = calculate_regime(df_processed)

    print("Processed data head:")
    display(df_processed.tail()) # Show the latest part of the processed data
    print(f"Total data points after processing: {len(df_processed)}")

except Exception as e:
    print(f"An error occurred during data download or processing: {e}")
    df_processed = pd.DataFrame() # Ensure df_processed is defined even on error


Fetching QQQ data from 2024-05-23 00:00:00 to 2026-05-23...


/tmp/ipykernel_3942/1360137593.py:64: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date, interval="1d")
[*********************100%***********************]  1 of 1 completed

Data downloaded successfully.
An error occurred during data download or processing: Data must be 1-dimensional, got ndarray of shape (501, 1) instead


## 3. Real-time Signal Judgment

In [18]:
import tensorflow as tf # Assuming a Keras/TensorFlow model for predictions
from sklearn.preprocessing import StandardScaler # Assuming feature scaling was used
import joblib # To load scaler or model if saved

# --- Load pre-trained model and scaler if applicable ---
# If your strategy uses a machine learning model, load it here.
# Otherwise, your signal generation logic will directly use indicators and parameters.

model = None # Placeholder for your loaded model
scaler = None # Placeholder for your loaded scaler

# Example: try to load a model and scaler
try:
    # Replace with your actual model and scaler paths
    # model = tf.keras.models.load_model('/content/drive/MyDrive/optuna_trading/my_trained_model.h5')
    # scaler = joblib.load('/content/drive/MyDrive/optuna_trading/my_scaler.pkl')
    print("Model and scaler loading (if applicable) - Remember to update paths.")
except Exception as e:
    print(f"Could not load model or scaler (expected if not using one, or if paths are incorrect): {e}")


# --- Signal Generation Logic (MUST MATCH BACKTEST) ---
def generate_signal(latest_bar_features, best_params, model, scaler):
    """
    Generates a trading signal (buy, sell, no trade) for the latest bar.
    This function *must* exactly match the signal generation logic used during backtesting.
    It should use the optimized 'best_params' and, if applicable, the loaded 'model' and 'scaler'.
    No future leakage is allowed.
    """
    signal = 'no trade'
    probability = 0.5 # Placeholder

    # --- IMPORTANT: Replace this with your actual signal generation logic ---
    # This example is a placeholder. Your actual logic will use 'latest_bar_features',
    # 'best_params', and potentially your 'model' and 'scaler'.

    if not best_params:
        print("Warning: best_params not loaded. Using default/example logic.")
        # Example logic if params are missing
        if latest_bar_features['RSI'].iloc[-1] < 30:
            signal = 'buy'
            probability = 0.7
        elif latest_bar_features['RSI'].iloc[-1] > 70:
            signal = 'sell'
            probability = 0.7
    else:
        # Example logic using best_params (replace with your actual strategy)
        # For instance, if your strategy threshold is in best_params
        rsi_buy_threshold = best_params.get('rsi_buy_threshold', 30)
        rsi_sell_threshold = best_params.get('rsi_sell_threshold', 70)

        if latest_bar_features['RSI'].iloc[-1] < rsi_buy_threshold:
            signal = 'buy'
            probability = 0.75 # Placeholder, derive from model output if using ML
        elif latest_bar_features['RSI'].iloc[-1] > rsi_sell_threshold:
            signal = 'sell'
            probability = 0.75

        # If using a model:
        # if model and scaler:
        #     features_scaled = scaler.transform(latest_bar_features[your_feature_columns].values.reshape(1, -1))
        #     prediction = model.predict(features_scaled)[0][0]
        #     probability = prediction
        #     signal_threshold = best_params.get('signal_probability_threshold', 0.5)
        #     if prediction > signal_threshold: signal = 'buy'
        #     elif prediction < (1 - signal_threshold): signal = 'sell'
        #     else: signal = 'no trade'

    return signal, probability

# --- Get the latest bar's data ---
if not df_processed.empty:
    latest_bar = df_processed.iloc[-1:] # Only the very last row
    current_regime = latest_bar['Regime'].iloc[0]

    print(f"Current market regime: {current_regime}")

    # Generate signal for the latest bar
    # Ensure 'latest_bar' contains all features needed for your 'generate_signal' function
    current_signal, signal_probability = generate_signal(latest_bar, best_params, model, scaler)

    print(f"Signal for current bar: {current_signal.upper()} (Probability: {signal_probability:.2f})")
else:
    print("Error: No processed data available to generate a signal.")
    current_regime = 'Unknown'
    current_signal = 'no trade'
    signal_probability = 0.0


Model and scaler loading (if applicable) - Remember to update paths.
Error: No processed data available to generate a signal.


## 4. Paper Trade Management

In [19]:
import datetime

class TradeManager:
    def __init__(self, initial_capital=100000, commission_rate=0.001, slippage_rate=0.0005, best_params=None):
        self.initial_capital = initial_capital
        self.capital = initial_capital
        self.position_size = 0 # Number of shares
        self.entry_price = 0
        self.entry_date = None
        self.trade_log = []
        self.is_in_position = False
        self.commission_rate = commission_rate
        self.slippage_rate = slippage_rate
        self.best_params = best_params if best_params is not None else {} # Use provided params or empty dict

        # Get ATR window and multiplier from best_params, or use defaults
        self.atr_window = self.best_params.get('atr_window', 14)
        self.atr_multiplier_sl = self.best_params.get('atr_multiplier_sl', 1.5)
        self.atr_multiplier_tp = self.best_params.get('atr_multiplier_tp', 2.0)


    def _calculate_fees(self, price, shares):
        return (price * shares * self.commission_rate) + (price * shares * self.slippage_rate)

    def enter_position(self, date, price, current_regime, signal_prob, atr_value, signal_type):
        if self.is_in_position: # Prevent double entry
            print(f"Already in a position on {date}. Skipping entry.")
            return

        # --- Position Sizing (MUST MATCH BACKTEST) ---
        # Example: risk a fixed percentage of capital per trade, or use fixed share size
        # Replace this with your actual position sizing logic from backtest
        risk_per_trade_percent = self.best_params.get('risk_per_trade_percent', 0.01) # e.g., 1% of capital
        stop_loss_level_based_on_atr = price - (atr_value * self.atr_multiplier_sl) # Example SL

        if stop_loss_level_based_on_atr <= 0: # Prevent division by zero or nonsensical SL
            risk_amount = self.capital * risk_per_trade_percent
            self.position_size = int(risk_amount / price) # Fallback to fixed dollar amount if SL invalid
        else:
            risk_per_share = price - stop_loss_level_based_on_atr
            if risk_per_share <= 0: # Ensure risk_per_share is positive
                risk_amount = self.capital * risk_per_trade_percent
                self.position_size = int(risk_amount / price) # Fallback
            else:
                self.position_size = int((self.capital * risk_per_trade_percent) / risk_per_share)

        if self.position_size == 0:
            print(f"Not enough capital or risk too high to open a position on {date}.")
            return

        entry_cost = price * self.position_size
        fees = self._calculate_fees(price, self.position_size)

        if (entry_cost + fees) > self.capital:
            print(f"Not enough capital to enter position on {date}. Required: {entry_cost + fees:.2f}, Available: {self.capital:.2f}")
            self.position_size = 0 # Reset position_size if not enough capital
            return

        self.capital -= (entry_cost + fees)
        self.entry_price = price
        self.entry_date = date
        self.is_in_position = True
        print(f"Entered {signal_type.upper()} position on {date} at {price:.2f} with {self.position_size} shares. Capital remaining: {self.capital:.2f}")

        # Define SL/TP based on ATR at entry
        self.stop_loss = price - (atr_value * self.atr_multiplier_sl) if signal_type == 'buy' else price + (atr_value * self.atr_multiplier_sl)
        self.take_profit = price + (atr_value * self.atr_multiplier_tp) if signal_type == 'buy' else price - (atr_value * self.atr_multiplier_tp)


    def exit_position(self, exit_date, exit_price, current_regime, reason):
        if not self.is_in_position:
            print(f"Not in a position to exit on {exit_date}. Skipping exit.")
            return

        pnl = (exit_price - self.entry_price) * self.position_size
        exit_fees = self._calculate_fees(exit_price, self.position_size)
        net_pnl = pnl - exit_fees

        self.capital += (self.entry_price * self.position_size) + net_pnl # Add back initial cost + net PnL

        trade_duration = (exit_date - self.entry_date).days

        self.trade_log.append({
            'entry_date': self.entry_date,
            'entry_price': self.entry_price,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'pnl': net_pnl,
            'holding_bars': trade_duration,
            'regime_at_entry': current_regime, # Log regime at entry for more completeness
            'signal_probability': signal_probability, # Added to log the signal probability at entry
            'stop_loss_hit': reason == 'SL' or reason == 'SL_Close',
            'take_profit_hit': reason == 'TP' or reason == 'TP_Close',
            'exit_reason': reason
        })

        print(f"Exited position on {exit_date} at {exit_price:.2f}. PnL: {net_pnl:.2f}. Capital: {self.capital:.2f}. Reason: {reason}")
        self.is_in_position = False
        self.position_size = 0
        self.entry_price = 0
        self.entry_date = None
        self.stop_loss = 0
        self.take_profit = 0


    def check_for_exit_intrabar(self, current_date, open_price, high_price, low_price, close_price, current_regime, next_signal='no trade'):
        if not self.is_in_position:
            return

        # --- Intrabar TP/SL (MUST MATCH BACKTEST) ---
        # Assume buy position first, then generalize
        if self.position_size > 0: # Long position
            # Check Take Profit
            if high_price >= self.take_profit and self.take_profit > self.entry_price: # Check if TP was hit and it's a profit
                self.exit_position(current_date, self.take_profit, current_regime, 'TP')
                return
            # Check Stop Loss
            if low_price <= self.stop_loss and self.stop_loss < self.entry_price: # Check if SL was hit and it's a loss
                self.exit_position(current_date, self.stop_loss, current_regime, 'SL')
                return
            # Check if next signal is 'sell' and force exit for strategy alignment
            if next_signal == 'sell':
                self.exit_position(current_date, close_price, current_regime, 'Signal_Exit_Sell')
                return

        elif self.position_size < 0: # Short position
            # Check Take Profit
            if low_price <= self.take_profit and self.take_profit < self.entry_price: # Check if TP was hit and it's a profit
                self.exit_position(current_date, self.take_profit, current_regime, 'TP')
                return
            # Check Stop Loss
            if high_price >= self.stop_loss and self.stop_loss > self.entry_price: # Check if SL was hit and it's a loss
                self.exit_position(current_date, self.stop_loss, current_regime, 'SL')
                return
            # Check if next signal is 'buy' and force exit for strategy alignment
            if next_signal == 'buy':
                self.exit_position(current_date, close_price, current_regime, 'Signal_Exit_Buy')
                return

        # If not exited intrabar by TP/SL or opposite signal, check at close price for SL/TP
        if self.is_in_position: # Still in position after intrabar checks
            if self.position_size > 0: # Long
                if close_price <= self.stop_loss:
                    self.exit_position(current_date, self.stop_loss, current_regime, 'SL_Close')
                elif close_price >= self.take_profit:
                    self.exit_position(current_date, self.take_profit, current_regime, 'TP_Close')
            elif self.position_size < 0: # Short
                if close_price >= self.stop_loss:
                    self.exit_position(current_date, self.stop_loss, current_regime, 'SL_Close')
                elif close_price <= self.take_profit:
                    self.exit_position(current_date, self.take_profit, current_regime, 'TP_Close')


# Initialize TradeManager
trade_manager = TradeManager(initial_capital=100000, commission_rate=best_params.get('commission_rate', 0.001), slippage_rate=best_params.get('slippage_rate', 0.0005), best_params=best_params)

print(f"Trade Manager Initialized with Capital: ${trade_manager.capital:.2f}")

# --- Simulate one day's trade management for the latest bar ---
# This section simulates how the strategy would behave given the latest completed day's data.
# The signal 'current_signal' is for the *next* trading day's entry.

if not df_processed.empty:
    # The latest_bar represents the *most recently completed day*.
    # The 'current_signal' derived from it is for potential entry on the *next day's open*.

    today_data = df_processed.iloc[-1] # This is the *latest completed* day
    today_date = today_data.name.date()
    today_open = today_data['Open']
    today_high = today_data['High']
    today_low = today_data['Low']
    today_close = today_data['Close']
    today_atr = today_data['ATR']
    today_regime = today_data['Regime']

    # First, check for intrabar exits if already in a position from yesterday's signal.
    # This means a position was opened at *today's open* based on *yesterday's close's signal*.
    # The intrabar checks use today's OHLC.
    trade_manager.check_for_exit_intrabar(today_date, today_open, today_high, today_low, today_close, today_regime, next_signal=current_signal)

    # If not currently in a position after checking for exits today, then decide on entering a new one.
    # This new entry would be for the *next trading day's open*, based on the 'current_signal' (from the latest completed bar).
    # For this simulation, we consider the signal from the latest completed bar ('today_data') as the signal for the 'next day'.
    # So, we'll simulate an entry at 'next_day_open' which for this real-time check, we don't have.
    # Therefore, this part needs to clarify the 'next bar' logic for real-time.
    # The request states '当日のCloseを見て当日約定しない', '次バー約定ルール'.
    # This means the 'current_signal' (based on `df_processed.iloc[-1]`) would trigger an entry on the *next day's open*.
    # Since we are doing a 'real-time near' simulation, we don't have tomorrow's open yet.
    # So, this part will just print the potential action for the *next* bar.

    if not trade_manager.is_in_position:
        if current_signal == 'buy':
            print(f"Strategy recommends BUY at the OPEN of the NEXT trading day, based on current conditions.")
        elif current_signal == 'sell':
            print(f"Strategy recommends SELL at the OPEN of the NEXT trading day, based on current conditions.")
        else:
            print(f"Strategy recommends NO TRADE for the NEXT trading day, based on current conditions.")
    else:
        print(f"Currently in an active position. No new entry decision for the next bar.")

    print("--- Trade Log (Last 5 Entries) ---")
    for trade in trade_manager.trade_log[-5:]:
        print(trade)
else:
    print("Cannot perform paper trade management due to missing processed data.")


Trade Manager Initialized with Capital: $100000.00
Cannot perform paper trade management due to missing processed data.


## 5. Performance Display

In [20]:
import numpy as np

def calculate_performance_metrics(trade_log, initial_capital):
    if not trade_log:
        return {
            'cumulative_return': 0,
            'sharpe_ratio': 0,
            'max_drawdown': 0,
            'profit_factor': 0,
            'win_rate': 0,
            'total_trades': 0
        }

    df_trades = pd.DataFrame(trade_log)

    # Ensure PnL is numeric
    df_trades['pnl'] = pd.to_numeric(df_trades['pnl'])

    # Cumulative Return (using final capital, or sum of PnL)
    total_pnl = df_trades['pnl'].sum()
    cumulative_return = total_pnl / initial_capital

    # Equity Curve for Sharpe and Drawdown
    equity_curve_values = [initial_capital]
    current_equity = initial_capital
    for pnl in df_trades['pnl']:
        current_equity += pnl
        equity_curve_values.append(current_equity)
    equity_curve = pd.Series(equity_curve_values)

    # Daily Returns from Equity Curve for Sharpe Ratio
    # This assumes the PnL entries are sequential in time.
    # For a true daily Sharpe, you would need daily equity data, not just trade-by-trade.
    # Here, we'll approximate using returns between trades.
    if len(equity_curve) > 1:
        returns = equity_curve.pct_change().dropna()
        # Annualize by roughly number of trading days, if trades are infrequent this is an approximation
        # More accurately, need to align PnL to calendar days.
        annualization_factor = 252 # Typical trading days in a year
        if len(returns) > 0 and returns.std() > 0:
            sharpe_ratio = np.mean(returns) / np.std(returns) * np.sqrt(annualization_factor)
        else:
            sharpe_ratio = 0
    else:
        sharpe_ratio = 0

    # Max Drawdown
    if not equity_curve.empty:
        peak = equity_curve.expanding(min_periods=1).max()
        drawdown = (equity_curve - peak) / peak
        max_drawdown = drawdown.min() * -1
    else:
        max_drawdown = 0

    # Profit Factor
    gross_profit = df_trades[df_trades['pnl'] > 0]['pnl'].sum()
    gross_loss = abs(df_trades[df_trades['pnl'] < 0]['pnl'].sum())
    profit_factor = gross_profit / gross_loss if gross_loss != 0 else np.inf

    # Win Rate
    total_trades = len(df_trades)
    winning_trades = len(df_trades[df_trades['pnl'] > 0])
    win_rate = winning_trades / total_trades if total_trades != 0 else 0

    return {
        'cumulative_return': cumulative_return,
        'sharpe_ratio': sharpe_ratio,
        'max_drawdown': max_drawdown,
        'profit_factor': profit_factor,
        'win_rate': win_rate,
        'total_trades': total_trades
    }

# Calculate and display performance
if trade_manager.trade_log:
    performance_metrics = calculate_performance_metrics(trade_manager.trade_log, trade_manager.initial_capital)

    print("--- Paper Trade Performance ---")
    for metric, value in performance_metrics.items():
        if isinstance(value, float):
            print(f"  {metric.replace('_', ' ').title()}: {value:.2f}")
        else:
            print(f"  {metric.replace('_', ' ').title()}: {value}")
else:
    print("No trades recorded yet. Cannot calculate performance.")


No trades recorded yet. Cannot calculate performance.


## 6. Real-time Operational Safety Check

In [21]:

# --- Operational Safety Checks (MUST MATCH BACKTEST OR REAL-WORLD RULES) ---
def conduct_safety_checks(df_processed, trade_manager, current_regime):
    safety_report = {}

    if df_processed.empty:
        safety_report['data_availability'] = 'No processed data. Cannot perform safety checks.'
        return safety_report

    # Regime Distribution (requires a historical view, assuming df_processed covers enough history)
    regime_counts = df_processed['Regime'].value_counts(normalize=True)
    safety_report['regime_distribution'] = regime_counts.to_dict()

    # Average Hold Time
    if trade_manager.trade_log:
        holding_bars = [trade['holding_bars'] for trade in trade_manager.trade_log if trade['holding_bars'] is not None]
        safety_report['average_hold_time_days'] = np.mean(holding_bars) if holding_bars else 0
    else:
        safety_report['average_hold_time_days'] = 0

    # Loss Streak (simplified - needs proper equity curve with trade results in sequence)
    loss_streaks = 0
    current_streak = 0
    for trade in trade_manager.trade_log:
        if trade['pnl'] < 0:
            current_streak += 1
        else:
            if current_streak > 0: # Only count if there was a streak
                loss_streaks = max(loss_streaks, current_streak)
            current_streak = 0
    loss_streaks = max(loss_streaks, current_streak) # Check for streak ending at the end
    safety_report['longest_loss_streak'] = loss_streaks

    # Volatility Condition (using current ATR relative to historical ATR)
    if 'ATR' in df_processed.columns and len(df_processed) > 30:
        current_atr = df_processed['ATR'].iloc[-1]
        historical_atr_mean = df_processed['ATR'].iloc[:-1].mean()
        historical_atr_std = df_processed['ATR'].iloc[:-1].std()

        if historical_atr_std > 0:
            z_score_atr = (current_atr - historical_atr_mean) / historical_atr_std
            safety_report['current_volatility_condition'] = f"ATR Z-score: {z_score_atr:.2f} (relative to historical mean)"
            if z_score_atr > 2: # Example threshold for high volatility
                safety_report['volatility_alert'] = 'High Volatility Detected'
            elif z_score_atr < -1: # Example threshold for low volatility
                safety_report['volatility_alert'] = 'Low Volatility Detected'
            else:
                safety_report['volatility_alert'] = 'Normal Volatility'
        else:
            safety_report['current_volatility_condition'] = 'Historical ATR std dev is zero, cannot calculate Z-score.'
            safety_report['volatility_alert'] = 'Normal Volatility (constant ATR)'
    else:
        safety_report['current_volatility_condition'] = 'ATR data not available or insufficient history.'
        safety_report['volatility_alert'] = 'N/A'


    # Whether current market is sideways (based on regime or other indicators)
    if current_regime == 'Sideways': # Requires your calculate_regime function to identify 'Sideways'
        safety_report['is_current_market_sideways'] = True
    else:
        # More robust check: e.g., low ADX, price contained within a narrow range
        if 'ADX' in df_processed.columns and df_processed['ADX'].iloc[-1] < 25: # Example threshold for ADX
            safety_report['is_current_market_sideways'] = True
            safety_report['sideways_reason'] = 'Low ADX'
        else:
            safety_report['is_current_market_sideways'] = False
            safety_report['sideways_reason'] = 'Not detected as sideways by current logic'

    return safety_report

# Conduct and display safety checks
safety_report = conduct_safety_checks(df_processed, trade_manager, current_regime)

print("--- Operational Safety Report ---")
for check, value in safety_report.items():
    if isinstance(value, dict):
        print(f"  {check.replace('_', ' ').title()}:")
        for k, v in value.items():
            print(f"    - {k}: {v:.2f}" if isinstance(v, float) else f"    - {k}: {v}")
    else:
        print(f"  {check.replace('_', ' ').title()}: {value}")


--- Operational Safety Report ---
  Data Availability: No processed data. Cannot perform safety checks.


## 7. Comprehensive Real-time Entry Decision

In [22]:
# --- Final Comprehensive Judgment ---
def comprehensive_entry_decision(current_signal, signal_probability, trade_manager, safety_report, best_params):
    decision = "No immediate entry recommendation."
    reasons = []

    if trade_manager.is_in_position:
        decision = "Currently in an active position. No new entry recommended until existing position is closed."
        reasons.append("Already in position.")
        return decision, reasons

    if current_signal == 'no trade':
        decision = "The strategy's signal for the next bar is 'no trade'."
        reasons.append("Strategy signal is 'no trade'.")
        return decision, reasons

    # Check signal strength (probability threshold from best_params)
    probability_threshold = best_params.get('signal_probability_threshold', 0.6) # Default 60%
    if signal_probability < probability_threshold:
        decision = f"Signal probability ({signal_probability:.2f}) is below the required threshold ({probability_threshold:.2f}). No entry."
        reasons.append("Signal probability too low.")
        return decision, reasons

    # Check safety conditions
    if safety_report.get('volatility_alert') == 'High Volatility Detected':
        decision = "High volatility detected. Proceed with caution or avoid entry."
        reasons.append("High volatility.")
    elif safety_report.get('volatility_alert') == 'Low Volatility Detected':
        decision = "Low volatility detected. May indicate sideways market. Proceed with caution or avoid entry."
        reasons.append("Low volatility.")

    if safety_report.get('is_current_market_sideways'):
        decision = "Market is currently detected as sideways. Trading in sideways markets can be challenging. Exercise caution."
        reasons.append("Sideways market detected.")

    # If we reached here, and signal is 'buy' or 'sell' with sufficient probability, it's a potential entry
    if current_signal in ['buy', 'sell'] and signal_probability >= probability_threshold:
        if not reasons: # If no negative safety checks triggered
            decision = f"**STRONGLY CONSIDER {current_signal.upper()} ENTRY on the next bar.** All conditions appear favorable."
        else:
            decision = f"**CONSIDER {current_signal.upper()} ENTRY on the next bar, but be aware of:** {', '.join(reasons)}."

    return decision, reasons


final_decision, decision_reasons = comprehensive_entry_decision(current_signal, signal_probability, trade_manager, safety_report, best_params)

print("\n--- REAL-TIME ENTRY DECISION ---")
print(final_decision)
if decision_reasons:
    print("Reasons/Considerations:")
    for r in decision_reasons:
        print(f"- {r}")



--- REAL-TIME ENTRY DECISION ---
The strategy's signal for the next bar is 'no trade'.
Reasons/Considerations:
- Strategy signal is 'no trade'.
